In [1]:
from typing import Optional
from openai import AsyncOpenAI
import os
import json
from pathlib import Path

In [2]:
print(os.environ.get('OPENAI_API_KEY'))

voc-3655732192140376144376a895519cc8d44.30553831


In [7]:
client = AsyncOpenAI()

MODEL = 'gpt-4o-mini'
JUDGE_MODEL = 'gpt-4o'
TEMPERATURE = 0.0
RATES = {
    'gpt-4o-mini': {'in': 0.15 / 1_000_000, 'out': 0.60 / 1_000_000},
    'gpt-4o': {'in': 2.50 / 1_0000_000, 'out': 10.00 / 1_000_000}
}

print('Setup completed!')

Setup completed!


In [4]:
DATA_DIR = Path('../data')   # adjust if your folder layout differs

snippets = [json.loads(line) for line in (DATA_DIR / 'job_snippets.jsonl').read_text().splitlines() if line.strip()]
golden = {row['id']: row for row in (json.loads(line) for line in (DATA_DIR / 'golden_set.jsonl').read_text().splitlines() if line.strip())}

print(f'Loaded {len(snippets)} snippets, {len(golden)} golden entries.')
print('Sample snippet:', snippets[0])
print('Golden entries:', golden['j01'])

Loaded 10 snippets, 10 golden entries.
Sample snippet: {'id': 'j01', 'snippet': 'Acme Corp is hiring a Senior Software Engineer to join our platform team. The ideal candidate has 5+ years of backend development experience and strong skills in Python and distributed systems.'}
Golden entries: {'id': 'j01', 'company': 'Acme Corp', 'role': 'Senior Software Engineer', 'years_experience_required': 5, 'notes': 'clean — clear company + role + years'}


In [20]:
def prompt_zero_shot(snippet_text: str) -> list[dict]:
    """Strategy 1 — zero-shot. Just ask, no examples, no persona."""
    content = (
        "Get the string fields company, role and years_experience as integer from the given job snippet text.\n"
        "If the snippet doesn't have years of experience required mentioned, then fill with null.\n"
        "Return the results in JSON format with above fields only.\n"
        f"job_snippet = {snippet_text}"
    )
    return [{"role": "user", "content": content}]


def prompt_few_shot(snippet_text: str) -> list[dict]:
    """Strategy 2 — few-shot. Include 2-3 worked examples in the prompt."""
    content = (
        "TASK:\n"
        "1. Analyze the given job description snippet text.\n"
        "2. Extract the string fields company, role and years_experience as integer.\n"
        "3. If the snippet doesn't have years of experience required mentioned, then fill with null.\n\n"
        "FORMAT: Return the results in JSON format with above fields only.\n\n"
        "# Example 1:\n"
        'User Input: "Acme Corp is hiring a Senior Software Engineer to join our platform team. The ideal candidate has 5+ years of backend development experience and strong skills in Python and distributed systems."\n'
        "Output:\n"
        '{\n'
        '    "company": "Acme Corp",\n'
        '    "role": "Senior Software Engineer",\n'
        '    "years_experience": 5\n'
        '}\n\n'
        "# Example 2:\n"
        'User Input: "Stark Industries: Cybersecurity Analyst (mid-level). Three to five years in a SOC environment or equivalent. Familiarity with SIEM tools, incident response runbooks, and basic threat modelling."\n'
        "Output:\n"
        '{\n'
        '    "company": "Stark Industries",\n'
        '    "role": "Cybersecurity Analyst",\n'
        '    "years_experience": 3\n'
        '}\n\n'
        "# Example 3:\n"
        'User Input: "Hooli is looking for a Junior Frontend Developer. Fresh grads welcome — no prior experience required. We care about curiosity and willingness to learn. JavaScript, React, and CSS fundamentals expected."\n'
        "Output:\n"
        '{\n'
        '    "company": "Hooli",\n'
        '    "role": "Junior Frontend Developer",\n'
        '    "years_experience": null\n'
        '}\n\n'
        f"job_snippet = {snippet_text}"
    )
    return [{"role": "user", "content": content}]


def prompt_structured(snippet_text: str) -> list[dict]:
    """Strategy 3 — structured / role-based. Use a system prompt with a persona and explicit JSON schema."""
    content = (
        "TASK:\n"
        "1. You are an expert recruiter.\n"
        "2. Extract the string fields company, role and years_experience as integer.\n"
        "3. If the snippet doesn't have years of experience required mentioned, then fill with null.\n\n"
        "FORMAT: Return the results in JSON format with above fields.\n\n"
        "CONTEXT: For the user-provided snippet text, get the output in JSON format with fields company, role, years_experience only.\n\n"
        "USER INPUT:\n"
        f"job_snippet = {snippet_text}"
    )
    return [{"role": "user", "content": content}]


def prompt_cot(snippet_text: str) -> list[dict]:
    """Strategy 4 — chain-of-thought. Ask the model to reason before answering."""
    content = (
        "You are an expert recruiter. Read the job description from the given snippet text and extract the company name, job role, and years of experience required.\n\n"
        "To ensure accuracy, follow this strict reasoning process before producing the final JSON output.\n\n"
        "### Instructions:\n"
        "1. Analyze the input: read the job description carefully.\n"
        "2. Step 1: Identify the required JSON fields and extract:\n"
        "   - company: company name\n"
        "   - role: job role or title\n"
        "   - years_experience: years required; if not mentioned, use null\n"
        "3. Step 2: Normalize values and validate:\n"
        "   - 7+ years -> 7\n"
        "   - 3-5 years -> 3 (minimum value in the range)\n"
        "   - around 6 years -> 6\n"
        "4. Step 3: Format the above three fields into a single valid JSON object.\n\n"
        "### Example:\n"
        'Input: "Hooli is looking for a Junior Frontend Developer. Fresh grads welcome — 2+ experience required. We care about curiosity and willingness to learn. JavaScript, React, and CSS fundamentals expected."\n\n'
        "Reasoning Process:\n"
        "- Step 1: Identify the required JSON fields and extract\n"
        "  - company: Hooli\n"
        "  - role: Junior Frontend Developer\n"
        "  - years_experience: 2+ years\n"
        "- Step 2: Normalize years_experience: 2\n"
        "- Step 3: Ready for JSON mapping.\n\n"
        "JSON Output Mapping:\n"
        '{\n'
        '    "company": "Hooli",\n'
        '    "role": "Junior Frontend Developer",\n'
        '    "years_experience": 2\n'
        '}\n\n'
        "USER INPUT:\n"
        f"job_snippet = {snippet_text}"
    )
    return [{"role": "user", "content": content}]


STRATEGIES = {
    'zero_shot': prompt_zero_shot,
    'few_shot': prompt_few_shot,
    'structured': prompt_structured,
    'cot': prompt_cot,
}


## Step 3: Async Batching

Run all 10 snippets x $ strategies = 40 cells in parallel
Capture for each call: strategy, snippet_id, raw response, parsed extraction, cost, latency.

In [21]:
import asyncio
import time
import re


def parse_response(text: str) -> dict | None:
    """Try to parse a JSON object out of the model's response. Return None if it doesn't parse."""
    if not text:
        return None

    cleaned = text.strip()

    if cleaned.startswith('```'):
        match = re.search(r'```(?:json)?\s*(.*?)\s*```', cleaned, flags=re.IGNORECASE | re.DOTALL)
        if match:
            cleaned = match.group(1).strip()

    responses = [cleaned]
    if '{' in cleaned and '}' in cleaned:
        first = cleaned.find('{')
        last = cleaned.rfind('}')
        if first >= 0 and last > first:
            responses.append(cleaned[first:last + 1])

    for response in responses:
        try:
            parsed = json.loads(response)
            if isinstance(parsed, dict):
                return parsed
        except Exception:
            pass

    return None


async def run_one(strategy_name: str, snippet: dict) -> dict:
    """Run one strategy on one snippet. Return a dict with all the captured fields."""
    strategy_function = STRATEGIES[strategy_name]
    messages = strategy_function(snippet['snippet'])

    start = time.perf_counter()
    response = await client.chat.completions.create(
        model=MODEL,
        temperature=TEMPERATURE,
        messages=messages
    )
    end = time.perf_counter()

    raw_text = response.choices[0].message.content
    parsed = parse_response(raw_text)
    usage = response.usage

    prompt_cost = usage.prompt_tokens * RATES[MODEL]['in'] if usage else 0
    completion_cost = usage.completion_tokens * RATES[MODEL]['out'] if usage else 0

    return {
        'strategy': strategy_name,
        'snippet_id': snippet['id'],
        'raw_response': raw_text,
        'parsed_extraction': parsed,
        'cost_usd': round(prompt_cost + completion_cost, 8),
        'latency_s': round(end - start, 3)
    }


async def run_all() -> list[dict]:
    """Run all 10 × 4 = 40 calls in parallel. Use asyncio.gather."""
    tasks = [
        run_one(strategy_name, snippet)
        for strategy_name in STRATEGIES
        for snippet in snippets
    ]
    return await asyncio.gather(*tasks)


In [22]:
# execute run_all()
results = await run_all()
print(f'Got {len(results)} results.')
results[0]

Got 40 results.


{'strategy': 'zero_shot',
 'snippet_id': 'j01',
 'raw_response': '```json\n{\n  "company": "Acme Corp",\n  "role": "Senior Software Engineer",\n  "years_experience": 5\n}\n```',
 'parsed_extraction': {'company': 'Acme Corp',
  'role': 'Senior Software Engineer',
  'years_experience': 5},
 'cost_usd': 3.39e-05,
 'latency_s': 0.873}

## Step 4 — Score against the golden set

Three scores per (strategy × snippet) pair:

1. **accuracy** — how many of 3 fields match (0, 1, 2, or 3)?
2. **parse_success** — did the response parse cleanly?
3. **llm_judge_score** — 1-4 score from gpt-4o-as-judge

In [27]:
def score_accuracy(extracted: dict | None, gold: dict) -> int:
    """Compare 3 fields. Case-insensitive, whitespace-trimmed for strings. Return 0, 1, 2, or 3."""
    # TODO: count exact matches (with normalisation)

    def norm(value):
        if value is None:
            return None
        if isinstance(value, str):
            return ' '.join(value.strip().lower().split())
        if isinstance(value, (int, float)):
            return int(value)
        return value
    
    score = 0
    for field in ['company', 'role', 'years_experience']:
        gold_value = norm(gold.get(field))
        extracted_value = norm(extracted.get(field))
        if gold_value == extracted_value:
            score += 1
    
    return score

async def score_llm_judge(snippet_text: str, extracted: dict | None, gold: dict) -> int:
    """Use gpt-4o as a judge. Return integer 1-4.
    
    Rubric (suggested):
      4 — all three fields correct
      3 — two of three correct, no fabricated data
      2 — one of three correct, or fabricated a field
      1 — none correct or unparsable
    """
    # TODO: prompt the judge with both the gold and the extracted, ask for a 1-4 score
    prompt = (
        "You are a judge and expert in evaluating the extracted job metadata.\n"
        "Score the model's output extracted against the expected gold attribute values using below rubric:\n"
        "4 — all three fields correct\n"
        "3 — two of three correct, no fabricated data\n"
        "2 — one of three correct, or fabricated a field\n"
        "1 — none correct or unparsable\n\n"
        f"Job snippet:\n{snippet_text}, \n"
        f"Gold answer:\n{json.dumps(gold)}\n"
        f"Model output:\n{json.dumps(extracted) if isinstance(extracted, dict) else extracted}\n\n"
        "Return only a single integet score from 1 to 4."
    )

    response = await client.chat.completions.create(
        model=JUDGE_MODEL,
        temperature=TEMPERATURE,
        messages=[{"role": "user", "content": prompt}]
    )

    text = response.choices[0].message.content.strip()
    match = re.search(r'\b([1-4])\b', text)
    return int(match.group(1)) if match else 1

In [28]:
scored = []
for item in results:
    snippet_id = item['snippet_id']
    snippet = next(s for s in snippets if s['id'] == snippet_id)
    # print(snippet)
    gold = golden[snippet_id]

    row = dict(item)
    row['accuracy'] = score_accuracy(item['parsed_extraction'], gold)
    row['parse_success'] = 1 if item['parsed_extraction'] is not None else 0
    row['llm_judge_score'] = await score_llm_judge(snippet['snippet'], item['parsed_extraction'], gold)
    scored.append(row)

print(f"Scored {len(scored)} results.")

Scored 40 results.


## Step 5 — Build the comparison table

In [35]:
import pandas as pd

df = pd.DataFrame(scored)
# print(df.head())

summary = df.groupby('strategy').agg({
        'accuracy': 'mean',
        'parse_success': 'mean',
        'llm_judge_score': 'mean',
        'cost_usd': 'sum',
        'latency_s': 'median'
    }).round(3)

summary.columns = ['Accuracy (mean)', 'Parse rate', 'Judge score', 'Cost ($)', 'Latency p50 (s)']
summary


,Accuracy (mean),Parse rate,Judge score,Cost ($),Latency p50 (s)
strategy,,,,,
cot,2.0,1.0,3.9,0.002,1.769
few_shot,2.1,1.0,3.9,0.001,1.162
structured,2.3,1.0,3.7,0.000,0.962
zero_shot,2.2,1.0,3.8,0.000,0.898
